# Emulation part 2: Partial Least Squares (PLS)

In the previous emulator notebook we used Data Space Inversion (DSI). DSI builds a purely statistical surrogate of the model *outputs* (it never looks at the model parameters). Here we explore a different flavour of emulator: **Partial Least Squares (PLS) regression**.

PLS finds a small set of latent factors that maximise the covariance between an input matrix `X` (the model **parameters**) and an output matrix `Y` (the model **observations**). Unlike DSI, a PLS emulator is an explicit, fast-running surrogate of the *parameter-to-observation* mapping. That means:

- the emulator's "parameters" are the actual model parameters, so we can condition the *real* parameters with PEST++-IES and inspect them afterwards;
- it is a natural fit for surrogate problems where the parameter dimension `d` is much larger than the number of training realizations `n` (exactly our situation here: ~3900 parameters, a few hundred realizations);
- because the outputs are correlated multivariate quantities, the latent factors capture the dominant joint behaviour with only a handful of components.

Just like DSI, the only time we need to run the (expensive) numerical model is to generate the training ensemble. Once the PLS emulator is fit, the "forward run" is a couple of matrix multiplies.

> **Note:** The `PLS` emulator is brand new and (for now) only available in the local development version of `pyemu`. The first code cell below explicitly points at that local repo so this notebook uses it. Once the feature is released this step can be dropped.

Same as it ever was... let'z'go!

## Getting ready

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
import pyemu
import flopy
from pyemu.emulators import PLS
print("using pyemu from:", pyemu.__file__)

sys.path.insert(0, "..")
import herebedragons as hbd

Prepare the working directory. We re-use the dataset constructed in the ["freyberg ies"](../part2_06_ies/freyberg_ies_1_basics.ipynb) tutorial, exactly as the DSI notebook did. This is the prior Monte Carlo ensemble we will train the emulator on.

In [ ]:
# specify the temporary working folder
ies_d = os.path.join('master_ies_1a_pls')
if os.path.exists(ies_d):
    shutil.rmtree(ies_d)

org_t_d = os.path.join("..","part2_06_ies","master_ies_1a")
if not os.path.exists(org_t_d):
    raise Exception(f"you need to run the {org_t_d} notebook")
shutil.copytree(org_t_d,ies_d)

Load the `Pst` control file from the full-order (numerical) model and remind ourselves of the observations and predictions.

In [ ]:
pst_freyberg = pyemu.Pst(os.path.join(ies_d, "pest.pst"))
pst_freyberg

In [ ]:
# the forecast (prediction) quantities of interest
predictions = pst_freyberg.pestpp_options['forecasts'].split(',')
predictions

For plotting later, load the full-order model's prior and posterior observation ensembles. These are the results of the "real" IES run from the freyberg ies notebook - we will compare the PLS emulator against them.

In [ ]:
# full-order-model prior and posterior observation ensembles (for comparison plots)
oe_pr = pst_freyberg.ies.obsen0.copy()
oe_pt = pst_freyberg.ies.get("obsen",pst_freyberg.control_data.noptmax).copy()
oe_pr.shape

## Building the training data

A PLS emulator maps **parameters -> observations**. So our training data is the *joint* prior ensemble: the prior **parameter** ensemble (the inputs, `X`) lined up against the prior **observation** ensemble (the outputs, `Y`), realization by realization.

Start by loading the prior parameter ensemble. These columns are the emulator inputs.

In [ ]:
pe_pr = pst_freyberg.ies.paren0.copy()
pe_pr.shape

As in previous notebooks, the observation weights were "balanced for visibility" and do not reflect measurement noise. We included a `standard_deviation` column to describe noise. Let's set the weights to be the inverse of the noise standard deviation. This makes Phi interpretable as a goodness-of-fit measure (Phi should not drop much below the number of non-zero observations, otherwise we are over-fitting).

In [ ]:
obs = pst_freyberg.observation_data
obs.loc[obs.weight>0, ['obsval','weight','standard_deviation']].tail()

In [ ]:
obs.loc[obs.weight>0,"weight"] = 1.0 / obs.loc[obs.weight>0,"standard_deviation"]
assert obs.weight.sum()>0, "no non-zero obs weights found"

We are carrying a lot of zero-weighted observations along for the ride. We only need the non-zero-weighted observations (for conditioning), the forecasts (for the analysis), the historical/forecast time-series we want to plot, **and** the `hk` (hydraulic conductivity field) observations - we will use the latter to plot the emulated parameter fields, exactly like the IES notebooks do.

Drop everything else (the per-time head arrays, the inc/cum budget terms, and the time-difference obs) to keep the emulator output dimension manageable. Note this is the same set as the DSI notebook, *except* we keep the `hk` array obs so we can visualise the conditioned conductivity field.

In [ ]:
# drop obs to reduce the emulator output dimension.
# NB: we deliberately KEEP the npf_k_layer1 ("hk") array obs so we can plot
# the emulated hydraulic-conductivity field (like the IES notebooks).
drop_list = [f for f in pst_freyberg.instruction_files if f.startswith("hdslay")]
drop_list.append("inc.csv.ins")
drop_list.append("cum.csv.ins")
drop_list.extend([f for f in pst_freyberg.instruction_files if ".tdiff." in f])

for o in drop_list:
    pst_freyberg.drop_observations(os.path.join(ies_d,o),pst_path='.')

obsnmes = pst_freyberg.observation_data.obsnme.values
print(len(obsnmes), "emulator output observations;", pst_freyberg.nnz_obs, "non-zero-weighted")

Now assemble the joint training frame. Columns are the parameter names (inputs) followed by the retained observation names (outputs); rows are the realizations.

In [ ]:
par_names = pst_freyberg.parameter_data.parnme.tolist()
oe_pr_red = oe_pr.loc[:, obsnmes]

# inputs (parameters) | outputs (observations), aligned by realization
data = pd.concat([pe_pr, oe_pr_red], axis=1)
print("training data shape (n_reals, n_pars + n_obs):", data.shape)
data.iloc[:5, :3]

## Training the PLS emulator

`pyemu.emulators` is the entry point for all things emulation. To build a `PLS` emulator we hand it:

- the joint training `data`;
- `input_names` (the parameter columns) and `output_names` (the observation columns);
- optionally the full-order `pst`, which lets the emulator scrape parameter bounds/groups and observation metadata when it builds the PEST++ interface later.

We leave `n_components=None`, which tells PLS to **choose the number of latent factors by k-fold cross-validation** on the training data. (You can also set it explicitly if you already know a good value.)

In [ ]:
pls = PLS(pst=pst_freyberg,           # optional - used to scrape par/obs metadata
          data=data,
          input_names=par_names,
          output_names=list(obsnmes),
          n_components=None,           # None -> pick by cross-validation
          cv_folds=5,
          verbose=True)
pls.fit();

The cross-validation picked the number of components that minimised the k-fold RMSE. We can inspect the CV scores and the selected value.

In [ ]:
print("selected n_components:", pls.n_components)

cv = pd.Series(pls._cv_scores)
fig,ax = plt.subplots(1,1,figsize=(4,3))
ax.plot(cv.index, cv.values, "o-")
ax.axvline(pls.n_components, color="r", ls="--", label=f"selected = {pls.n_components}")
ax.set_xlabel("n_components")
ax.set_ylabel("cv rmse")
ax.set_xscale("log")
ax.legend()
fig.tight_layout()

To run the emulator directly, call `pls.predict()` with a set of parameter values. This is effectively the PLS "forward run". When we set up PEST++ below, IES will adjust these parameter values to improve the fit to observations. Let's sanity-check it on the first prior realization.

In [ ]:
pvals = pe_pr.iloc[[0]]   # a single set of parameter values
sim = pls.predict(pvals)
sim.head()

## Preparing the PEST++ interface

Just like the `DSI` object, the `PLS` object knows how to write out a complete PEST++ template directory. Calling `.prepare_pestpp()` pickles the fitted emulator, writes template/instruction files, a `forward_run.py` that loads the emulator and predicts, and a control file.

We pass `use_runstor=True`. This builds a forward run designed for PEST++-IES's **external run manager** (the `/e` flag): instead of shuffling CSV files to and from disk for every realization, PEST++ routes the whole ensemble through a single binary run-store that the forward run reads and writes in one shot. This is the fast way to drive a cheap emulator. We also give it `pst_name="pls"` so the forward run knows which run-store file to look for.

In [ ]:
t_d = "pls_template"
pst = pls.prepare_pestpp(t_d=t_d,
                         observation_data=pst_freyberg.observation_data,
                         use_runstor=True,
                         pst_name="pls")

`prepare_pestpp` does not copy executables, so grab the `pestpp-ies` binary from the original template directory.

In [ ]:
found = False
for f in os.listdir(ies_d):
    if f.startswith("pestpp-ies"):
        shutil.copy2(os.path.join(ies_d,f),os.path.join(t_d,f))
        found = True
if not found:
    raise Exception("couldn't find pestpp-ies binary in {0}".format(ies_d))
os.listdir(t_d)

Note the emulator parameters *are* the real model parameters - no dimension reduction on the parameter side, unlike DSI. Verify the parameter and observation counts.

In [ ]:
print("emulator npar_adj:", pst.npar_adj, " full-model npar_adj:", pst_freyberg.npar_adj)
print("emulator nobs    :", pst.nobs,     " emulator nnz_obs:", pst.nnz_obs)

Because the emulator inputs *are* the model parameters, we can condition IES with the **same prior parameter ensemble** that generated the training data. That makes the PLS-IES result directly comparable to the full-order-model IES result. Hand it to PEST++ via `ies_par_en`.

As with any emulator, it is a bad idea to extrapolate beyond the range of the training data, so we also switch on prior-data conflict resolution.

In [ ]:
# condition with the same prior parameter ensemble used for training
pe_pr.to_csv(os.path.join(t_d,"prior_pe.csv"))
pst.pestpp_options["ies_par_en"] = "prior_pe.csv"
pst.pestpp_options["ies_drop_conflicts"] = True

# number of realizations - as many as you can afford; here we use all the training reals
num_reals = pe_pr.shape[0]
pst.pestpp_options["ies_num_reals"] = num_reals

# set noptmax and write
pst.control_data.noptmax = 3
pst.write(os.path.join(t_d,"pls.pst"),version=2)

## Running IES with the external run manager

Now run PEST++-IES against the PLS emulator. Because we built the interface with `use_runstor=True`, we drive it with the **external run manager** by appending the `/e` flag. In this mode a single PEST++ process runs the whole ensemble through the emulator via the binary run-store - no parallel workers, no per-run disk I/O. For a fast surrogate like PLS this is plenty quick.

We copy the template into a fresh master directory and let it rip.

In [ ]:
m_d = "master_pls"
if os.path.exists(m_d):
    shutil.rmtree(m_d)
shutil.copytree(t_d, m_d)

# /e -> external (in-process) run manager, reading/writing the binary run-store
pyemu.os_utils.run("pestpp-ies pls.pst /e", cwd=m_d)

That was fast! Let's look at the evolution of the objective function (Phi). This is the same Phi - observations and weights - we have used throughout the data-assimilation notebooks. We want to see it decrease as the emulator is conditioned on the observations, but not drop much below the number of non-zero-weighted observations (`nnz_obs`), which would signal over-fitting.

In [ ]:
pst = pyemu.Pst(os.path.join(m_d,"pls.pst"))

phidf = pst.ies.phiactual

fig,ax=plt.subplots(1,1,figsize=(4,4))
ax.plot(phidf.index,phidf['mean'],"bo-", label='pls')

ax.set_yscale('log')
ax.set_ylabel('phi')
ax.set_xlabel('iteration')

ax.text(0.7,0.9,f"nnz_obs: {pst.nnz_obs}\nphi_pls: {phidf['mean'].iloc[-1]:.2f}",
        transform=ax.transAxes,ha="right",va="top")

ax.legend();

As usual, let's bring this back to the predictions. How did the PLS emulator do? We compare the emulator's prior and posterior forecast distributions against the full-order numerical model results from the "freyberg ies" notebooks.

In [ ]:
def plot_pls_hist(m_d,pst,iteration=3):
    pr_oe_pls = pst.ies.obsen0.copy()
    pt_oe_pls = pst.ies.get("obsen",iteration).copy()

    fig,axes = plt.subplots(len(predictions)//2,2,figsize=(7,7))
    for p,ax in zip(predictions,axes.flatten()):
            #calculate consistent bin edges
            bins = np.linspace(
                    min(oe_pr.loc[:,p].values.min(),
                        oe_pt.loc[:,p].values.min(),
                        pr_oe_pls.loc[:,p].values.min(),
                        pt_oe_pls.loc[:,p].values.min()),
                    max(oe_pr.loc[:,p].values.max(),
                        oe_pt.loc[:,p].values.max(),
                        pr_oe_pls.loc[:,p].values.max(),
                        pt_oe_pls.loc[:,p].values.max()),
                    30)

            ax.hist(oe_pr.loc[:,p].values,alpha=0.5,facecolor="0.5",density=True,label="prior",bins=bins)
            ax.hist(oe_pt.loc[:, p].values,  alpha=0.5, facecolor="b",density=True,label="posterior",bins=bins)
            ax.hist(pr_oe_pls.loc[:, p].values,  facecolor="none",hatch="/",edgecolor="0.5",lw=0.5,density=True,label="pls prior",bins=bins)
            ax.hist(pt_oe_pls.loc[:, p].values,  facecolor="none",density=True,hatch="/",edgecolor="b",lw=.5,label="pls posterior",bins=bins)

            fval = pst.observation_data.loc[p,"obsval"]
            ax.plot([fval,fval],ax.get_ylim(),"r-",label='truth')

            ax.set_title(p,loc="left")
            ax.legend(loc="best")
            ax.set_yticks([])
    fig.tight_layout()

In [ ]:
iteration = pst.control_data.noptmax
plot_pls_hist(m_d,pst,iteration=iteration)

The grey/blue solid histograms are the full-order-model prior/posterior; the hatched histograms are the PLS emulator prior/posterior; the red line is the truth. Where the emulator posterior (hatched blue) tightens up around the truth relative to its prior (hatched grey), the surrogate is successfully assimilating the data.

Now the forecast time series. The next function plots the posterior time-series ensemble for the historical and forecast periods (same plot as in the IES notebooks).

In [ ]:
def plot_tseries_ensembles(pt_oe, onames=["hds","sfr"],noise_oe=None):
    pst.try_parse_name_metadata()
    # get the observation data from the control file and select
    obs = pst.observation_data.copy()
    # onames provided in oname argument
    obs = obs.loc[obs.oname.apply(lambda x: x in onames)]
    # only non-zero observations
    obs = obs.loc[obs.obgnme.apply(lambda x: x in pst.nnz_obs_groups),:]
    # make a plot
    ogs = obs.obgnme.unique()
    fig,axes = plt.subplots(len(ogs),1,figsize=(7,2*len(ogs)))
    ogs = sorted(ogs)
    # for each observation group (i.e. timeseries)
    for ax,og in zip(axes,ogs):
        # get values for x axis
        oobs = obs.loc[obs.obgnme==og,:].copy()
        oobs["time"] = oobs["time"].astype(float)
        oobs.sort_values(by="time",inplace=True)
        tvals = oobs.time.values
        onames = oobs.obsnme.values
        ax.plot(oobs.time,oobs.obsval,color="fuchsia",lw=2,zorder=99)
        # plot posterior
        [ax.plot(tvals,pt_oe.loc[i,onames].values,"b",lw=0.5,alpha=0.5) for i in pt_oe.index]
        # plot measured+noise
        oobs = oobs.loc[oobs.weight>0,:]
        tvals = oobs.time.values
        onames = oobs.obsnme.values
        if noise_oe is not None:
            [ax.plot(tvals,noise_oe.loc[i,onames].values,"r",lw=0.5,alpha=0.5,zorder=0) for i in noise_oe.index]
        ax.plot(oobs.time,oobs.obsval,"r-o",lw=2,zorder=100)
        ax.set_title(og,loc="left")
    fig.tight_layout()
    return fig

In [ ]:
pt_oe_pls = pst.ies.get("obsen",iteration)
fig = plot_tseries_ensembles(pt_oe_pls, onames=["hds","sfr"])

## Emulated parameter fields

Here is where a PLS emulator earns its keep relative to DSI: because the emulator's inputs *are* the real model parameters, IES conditions those parameters directly. The emulator also predicts the `hk` array observations, so we can map the **emulated hydraulic-conductivity field** before and after conditioning - the same prior-vs-posterior `hk` plot used in the ["freyberg ies"](../part2_06_ies/freyberg_ies_1_basics.ipynb) notebooks.

First grab the model grid (for the active-cell mask and to place the `hk` obs back onto the grid).

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=ies_d, verbosity_level=0)
ib = sim.get_model().dis.idomain.array[0,:,:]

The `plot_hk` function below is lifted straight from the IES notebooks. It paints the prior and posterior `hk` ensembles back onto the grid and shows, for each, the ensemble mean, standard deviation, the "base"/most-likely realization (MEV), and a single example realization.

In [ ]:
def plot_hk(pr_oe, pt_oe, pst):
    pst.try_parse_name_metadata()
    obs = pst.observation_data
    hkobs = obs.loc[obs.oname=="hk",:].copy()
    hkobs["i"] = hkobs.i.astype(int)
    hkobs["j"] = hkobs.j.astype(int)
    real = pt_oe.index[0]

    fig,axes = plt.subplots(2,4,figsize=(15,10))
    prmn,prstd,prmev,prreal = np.zeros_like(ib,dtype=float),np.zeros_like(ib,dtype=float),np.zeros_like(ib,dtype=float),np.zeros_like(ib,dtype=float)
    prmn[hkobs.i,hkobs.j] = pr_oe.loc[:,hkobs.obsnme].mean()
    prstd[hkobs.i,hkobs.j] = pr_oe.loc[:,hkobs.obsnme].std()
    prmev[hkobs.i,hkobs.j] = pr_oe.loc["base",hkobs.obsnme].values
    prreal[hkobs.i,hkobs.j] = pr_oe.loc[real,hkobs.obsnme].values
    ptmn,ptstd,ptmev,ptreal = np.zeros_like(ib,dtype=float),np.zeros_like(ib,dtype=float),np.zeros_like(ib,dtype=float),np.zeros_like(ib,dtype=float)
    ptmn[hkobs.i,hkobs.j] = pt_oe.loc[:,hkobs.obsnme].mean()
    ptstd[hkobs.i,hkobs.j] = pt_oe.loc[:,hkobs.obsnme].std()
    ptmev[hkobs.i,hkobs.j] = pt_oe.loc["base",hkobs.obsnme].values
    ptreal[hkobs.i,hkobs.j] = pt_oe.loc[real,hkobs.obsnme].values
    for arr in [prmn,prstd,prmev,prreal,ptmn,ptstd,ptmev,ptreal]:
        arr[ib>0] = np.log10(np.abs(arr[ib>0]))
        arr[ib==0] = np.nan
    prarrs = [prmn,prstd,prmev,prreal]
    ptarrs = [ptmn,ptstd,ptmev,ptreal]
    titles = ["mean","stdev","MEV","realization {0}".format(real)]

    for pr,pt,axs,title in zip(prarrs,ptarrs,axes.transpose(),titles):
        vmn,vmx = min(np.nanmin(pr),np.nanmin(pt)),max(np.nanmax(pr),np.nanmax(pt))
        cb = axs[0].imshow(pr,vmin=vmn,vmax=vmx)
        plt.colorbar(cb,ax=axs[0],label="$log_{10}\\frac{m}{d}$")
        axs[0].set_title("prior "+title)
        cb = axs[1].imshow(pt,vmin=vmn,vmax=vmx)
        plt.colorbar(cb,ax=axs[1],label="$log_{10}\\frac{m}{d}$")
        axs[1].set_title("posterior "+title)
    plt.tight_layout()
    return fig,axes

In [ ]:
pr_oe_pls = pst.ies.obsen0.copy()
pt_oe_pls = pst.ies.get("obsen",iteration).copy()
_ = plot_hk(pr_oe_pls, pt_oe_pls, pst)

The top row is the prior (emulated) `hk` field, the bottom row the posterior. The standard-deviation panels (second column) tell the story: where the posterior std collapses relative to the prior, conditioning on the head and flow data has informed the conductivity field. Keep in mind these are the *emulator's* reconstruction of `hk` - a linear PLS surrogate - so fine-scale structure is smoothed relative to the full-order model, but the large-scale conditioned pattern comes through.

# Final remarks

We just trained a Partial Least Squares emulator of the Freyberg model's parameter-to-observation mapping and conditioned it with PEST++-IES using the external run manager - all off a few hundred prior model runs.

How PLS compares to DSI:

- **DSI** emulates the model *outputs* directly (parameters never enter), and conditions a latent set of surrogate parameters. It is great for pure predictive uncertainty quantification.
- **PLS** emulates the *parameter-to-observation* map, so IES adjusts the **real** model parameters. That means the conditioned parameter ensemble is in the original parameter space and can be inspected, fed forward to new scenarios, or compared directly to a full-order IES result.

### Things to keep in mind
- Use as many realizations as you can afford for the training ensemble - more is better, especially when the parameter dimension is large.
- Let cross-validation pick `n_components`, but sanity-check the CV curve; too many components over-fits the training data, too few under-fits.
- Watch for forecasts leaning toward (or beyond) the ends of the prior distribution - a red flag that the surrogate is extrapolating. Keep prior-data conflict resolution on.
- A linear PLS surrogate captures linear-ish parameter-to-observation relationships cheaply; strongly non-linear mappings may need transforms (the same transform plumbing the DSI emulator uses is available here) or a richer emulator.